# 05 · The OpenCV engine

OpenCV is a first-class part of Vibes Lite: cleaning, masking, enhancement,
morphology and contour extraction live in `backend/opencv.py`, independent of
the API layer.

Operations can be composed one by one (below) or as a pipeline with
`process_image` - exactly what the **Computer Vision** page calls.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## 1. Read a band as an image

In [ ]:
band = data.load_single_band(RASTER, 4)          # band 4 = NIR in the demo scene
nir = np.asarray(band["array"], dtype="float32")
print("keys    :", sorted(band))
print("shape   :", nir.shape, "| range: %.0f .. %.0f" % (nir.min(), nir.max()))

image = vl_cv.to_display(nir)                    # 3-channel uint8 for compositing
print("display :", image.shape, image.dtype)
print("stats   :", vl_cv.image_statistics(image))

## 2. Clean and enhance

In [ ]:
denoised = vl_cv.denoise(image, method="nlm")
enhanced = vl_cv.enhance_image(denoised, method="clahe")
print("denoise -> enhance applied | mean brightness %.1f -> %.1f" % (image.mean(), enhanced.mean()))

## 3. Masking: threshold, Otsu and adaptive

`create_mask` returns the mask plus the method and threshold it used.

In [ ]:
def mask_of(value):
    """create_mask/clean_mask return a dict; plain helpers return an array."""
    return value["mask"] if isinstance(value, dict) else value


otsu_result = vl_cv.create_mask(enhanced, method="otsu", invert=False)
print("otsu keys     :", sorted(otsu_result))
print("otsu value    :", otsu_result.get("threshold"))
print("mask coverage :", round(vl_cv.mask_coverage(mask_of(otsu_result)), 4))

adaptive_mask = mask_of(vl_cv.create_mask(enhanced, method="adaptive", block_size=35, constant=5))
print("adaptive cover:", round(vl_cv.mask_coverage(adaptive_mask), 4))

## 4. Morphology

Opening removes speckle, closing fills pinholes, and `clean_mask` combines both with a minimum-area filter - the standard clean-up before counting anything.

In [ ]:
binary = mask_of(otsu_result)
opened = vl_cv.morph_open(binary)
closed = vl_cv.morph_close(binary)
cleaned = vl_cv.clean_mask(binary, open_kernel=5, close_kernel=5, min_area_pixels=64, fill_holes=True)
cleaned_mask = mask_of(cleaned)

print("raw coverage     :", round(vl_cv.mask_coverage(binary), 4))
print("opened coverage  :", round(vl_cv.mask_coverage(opened), 4))
print("closed coverage  :", round(vl_cv.mask_coverage(closed), 4))
print("cleaned coverage :", round(vl_cv.mask_coverage(cleaned_mask), 4))
print("cleaned keys     :", sorted(cleaned))

## 5. Contours and an annotated overlay

In [ ]:
contours = vl_cv.detect_contours(cleaned_mask, mode="external", min_area_pixels=64)
print("contours found   :", len(contours))
if contours:
    print("largest area (px):", max(contour["area_pixels"] for contour in contours))
    print("contour keys     :", sorted(contours[0]))

annotated = vl_cv.annotate_contours(image, contours[:40], label_areas=True)
overlay = vl_cv.create_overlay(image, mask=cleaned_mask, contours=contours[:40], alpha=0.45, label="vegetation")
print("annotated:", annotated.shape, "| overlay:", overlay.shape)

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(11, 9))
panels = (("input (NIR)", image), ("enhanced (CLAHE)", enhanced), ("cleaned mask", cleaned_mask), ("contours", annotated))
for axis, (title, picture) in zip(axes.ravel(), panels):
    axis.imshow(picture if picture.ndim == 3 else picture, cmap=None if picture.ndim == 3 else "gray")
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 6. The same thing as a pipeline

`process_image` runs a list of operations (with optional parameters) and returns step-by-step metrics - the payload behind the Computer Vision page's before/after view.

In [ ]:
pipeline = vl_cv.process_image(
    image,
    operations=["grayscale", "denoise", "enhance", {"name": "otsu"}, {"name": "morph_open", "params": {"kernel": 3}}, "remove_small_objects"],
    detect_contours_flag=True,
)
print("keys           :", sorted(pipeline))
print("operations run :", [step["operation"] for step in pipeline["steps"]])
print("contours       :", pipeline["contour_count"])
print("mask coverage  :", round(vl_cv.mask_coverage(pipeline["binary"]), 4))
print("output         :", pipeline["image"].shape, pipeline["output_dtype"])

In [ ]:
after = pipeline["image"] if pipeline["image"].ndim == 3 else vl_cv.to_display(pipeline["image"])
side = vl_cv.side_by_side(image, after)
vl_cv.save_image(NOTEBOOK_OUT / "05_before_after.png", side)

figure, axes = plt.subplots(figsize=(12, 5))
axes.imshow(side)
axes.set_title("Before  |  After")
axes.axis("off")
plt.tight_layout()
plt.show()
print("saved:", (NOTEBOOK_OUT / "05_before_after.png").relative_to(ROOT))

**Operations are a registry**, so the API and the UI stay in sync:

```
adaptive_threshold`, `blur`, `clean_mask`, `denoise`, `dilate`, `edges`, `enhance`, `erode`, `grayscale`, `invert`, `median_blur`, `morph_close`, `morph_gradient`, `morph_open`, `otsu`, `remove_small_objects`, `resize`, `threshold
```

Every step is a plain function, so it can be reused in scripts, tests or a
future deep-learning pre-processing pipeline without touching the API.